# 01 — Exploratory Data Analysis (EDA)
**Dataset:** Tokopedia Product Reviews 2019  
**Tujuan:** Memahami karakteristik dataset sebelum melakukan ABSA & SNA.

---

In [ ]:
import os
import sys
# Auto-adjust CWD to project root if executed from notebooks directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_palette('Set2')
print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
import os

# Auto-detect path CSV
for candidate in ['data/raw/tokopedia-product-reviews-2019.csv',
                   'data/raw/tokopedia-product-reviews-2019.csv']:
    if os.path.exists(candidate):
        RAW_PATH = candidate
        break

df_raw = pd.read_csv(RAW_PATH)
print(f'Shape raw: {df_raw.shape}')
df_raw.head(3)

In [ ]:
print('Kolom:', df_raw.columns.tolist())
print()
df_raw.info()

## 2. Statistik Dasar

In [ ]:
# Load versi clean (setelah preprocessing.py dijalankan)
CLEAN_PATH = 'data/processed/reviews_clean.csv'
df = pd.read_csv(CLEAN_PATH)
print(f'Raw   : {len(df_raw):,} baris')
print(f'Clean : {len(df):,} baris ({len(df_raw)-len(df):,} dihapus)')
print()
df.describe(include='all')

## 3. Distribusi Rating Bintang

In [ ]:
if 'rating' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Count bar
    rating_counts = df['rating'].value_counts().sort_index()
    colors = ['#F44336','#FF9800','#FFC107','#8BC34A','#4CAF50']
    bars = axes[0].bar(rating_counts.index, rating_counts.values,
                       color=colors, edgecolor='white', linewidth=0.8)
    for b in bars:
        axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+50,
                     f'{int(b.get_height()):,}', ha='center', fontsize=9)
    axes[0].set_xlabel('Rating Bintang'); axes[0].set_ylabel('Jumlah Ulasan')
    axes[0].set_title('Distribusi Rating Bintang')

    # Pie
    axes[1].pie(rating_counts.values, labels=[f'★{i}' for i in rating_counts.index],
                colors=colors, autopct='%1.1f%%', startangle=90,
                wedgeprops=dict(edgecolor='white', linewidth=1.5))
    axes[1].set_title('Proporsi Rating')

    plt.suptitle('Distribusi Rating Ulasan Tokopedia', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/figures/eda_rating_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Kolom rating tidak ditemukan di dataset.')

## 4. Distribusi Panjang Teks

In [ ]:
df['text_len'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['text_len'], bins=50, color='#5C6BC0', edgecolor='white', linewidth=0.5)
axes[0].axvline(df['text_len'].median(), color='#E53935', linestyle='--',
                linewidth=2, label=f"Median: {df['text_len'].median():.0f} kata")
axes[0].axvline(df['text_len'].mean(), color='#FB8C00', linestyle='--',
                linewidth=2, label=f"Mean: {df['text_len'].mean():.1f} kata")
axes[0].set_xlabel('Panjang Teks (kata)'); axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Panjang Ulasan'); axes[0].legend()
axes[0].set_xlim(0, df['text_len'].quantile(0.99))

# Box plot by rating
if 'rating' in df.columns:
    df.boxplot(column='text_len', by='rating', ax=axes[1],
               patch_artist=True, 
               boxprops=dict(facecolor='#5C6BC0', alpha=0.6))
    axes[1].set_xlabel('Rating Bintang'); axes[1].set_ylabel('Panjang (kata)')
    axes[1].set_title('Panjang Ulasan per Rating')
    plt.suptitle('')

plt.tight_layout()
plt.savefig('results/figures/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()
print(df['text_len'].describe())

## 5. Kata Paling Sering Muncul

In [ ]:
from collections import Counter

STOPWORDS = {'yang','dan','di','ke','dari','dengan','untuk','ini','itu',
             'sudah','dengan','tidak','ada','saya','ya','aja','juga',
             'tapi','tapi','tapi','kalau','atau','bisa','nih','sih',
             'deh','lagi','banget','sangat','nya','pun','jadi','baru',
             'kita','kami','kamu','pada','se','ter','me','di','ke','dari'}

all_words = []
for text in df['text'].dropna():
    words = str(text).split()
    all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 2])

word_freq = Counter(all_words).most_common(30)
words_df = pd.DataFrame(word_freq, columns=['word','count'])

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(words_df['word'][::-1], words_df['count'][::-1],
               color='#26A69A', edgecolor='white')
for b in bars:
    ax.text(b.get_width()+20, b.get_y()+b.get_height()/2,
            f'{int(b.get_width()):,}', va='center', fontsize=9)
ax.set_xlabel('Frekuensi'); ax.set_title('30 Kata Paling Sering Muncul dalam Ulasan', fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/eda_word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Kata Kunci per Aspek

In [ ]:
import sys
sys.path.insert(0, 'src')
from absa_model import ASPECT_KEYWORDS

aspect_mention = {}
for aspect, keywords in ASPECT_KEYWORDS.items():
    count = df['text'].dropna().apply(
        lambda t: any(kw in t.lower() for kw in keywords)
    ).sum()
    aspect_mention[aspect] = count

asp_df = pd.DataFrame(list(aspect_mention.items()), columns=['Aspek', 'Jumlah Ulasan'])
asp_df = asp_df.sort_values('Jumlah Ulasan', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#EF5350','#AB47BC','#42A5F5','#26A69A','#FFA726']
bars = ax.bar(asp_df['Aspek'], asp_df['Jumlah Ulasan'], color=colors, edgecolor='white')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+30,
            f'{int(b.get_height()):,}', ha='center', fontsize=10)
ax.set_ylabel('Jumlah Ulasan')
ax.set_title('Jumlah Ulasan yang Menyebut Tiap Aspek (dari seluruh dataset)', fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/eda_aspect_mention.png', dpi=150, bbox_inches='tight')
plt.show()
asp_df

## 7. Ringkasan EDA

In [ ]:
print('=' * 50)
print('RINGKASAN EDA')
print('=' * 50)
print(f'Total ulasan (raw)   : {len(df_raw):,}')
print(f'Total ulasan (clean) : {len(df):,}')
print(f'Median panjang teks  : {df["text_len"].median():.0f} kata')
print(f'Rata-rata panjang    : {df["text_len"].mean():.1f} kata')
if 'rating' in df.columns:
    print(f'Rating dominan       : ★{df["rating"].mode()[0]}')
print()
print('Aspek paling banyak disebut:')
for _, row in asp_df.iterrows():
    pct = row['Jumlah Ulasan'] / len(df) * 100
    print(f"  {row['Aspek']:12s}: {row['Jumlah Ulasan']:,} ulasan ({pct:.1f}%)")